In [13]:
# Intialize Spark Session

In [4]:
from pyspark.sql import SparkSession 

spark = SparkSession.builder \
.appName('olist_dataset')\
.master('yarn')\
.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/05 08:45:17 INFO SparkEnv: Registering MapOutputTracker
26/03/05 08:45:17 INFO SparkEnv: Registering BlockManagerMaster
26/03/05 08:45:17 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/03/05 08:45:17 INFO SparkEnv: Registering OutputCommitCoordinator


In [5]:
spark

In [14]:
# Ingest All Bronze Layer Data and MongoDB Data into Dataproc

In [8]:
base_path = 'gs://otis-dataset-project/Bronze/'

customers_path = 'olist_customers_dataset.csv'

customers_df = spark.read \
                .format('csv') \
                .option('header','true') \
                .option('inferschema','true') \
                .load(base_path + customers_path)

In [10]:
customers_df.show()

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
|879864dab9bc30475...|4c93744516667ad3b...|                   89254|      jaragua do sul|            SC|
|fd826e7cf63160e53...|addec96d2e059c80c...|            

In [15]:
orders_path = base_path + "olist_orders_dataset.csv"
payments_path = base_path + "olist_order_payments.csv"
reviews_path = base_path + "olist_order_reviews_dataset.csv"
items_path = base_path + "olist_order_items_dataset.csv"
sellers_path = base_path + "olist_sellers_dataset.csv"
geolocation_path = base_path + "olist_geolocation_dataset.csv"
products_path = base_path + "olist_products_dataset.csv"

orders_df = spark.read.format("csv").option("header","true").option("inferschema","true").load(orders_path)
payments_df = spark.read.format("csv").option("header","true").option("inferschema","true").load(payments_path)
reviews_df = spark.read.format("csv").option("header","true").option("inferschema","true").load(reviews_path)
items_df = spark.read.format("csv").option("header","true").option("inferschema","true").load(items_path)
sellers_df = spark.read.format("csv").option("header","true").option("inferschema","true").load(sellers_path)
geolocation_df = spark.read.format("csv").option("header","true").option("inferschema","true").load(geolocation_path)
products_df = spark.read.format("csv").option("header","true").option("inferschema","true").load(products_path)

In [16]:
## Mongo DB Dataset Ingestion

In [18]:
!pip install pymongo

   â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 1.4/1.4 MB 20.2 MB/s eta 0:00:0000:010:01
   â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 331.1/331.1 kB 38.0 MB/s eta 0:00:00


In [ ]:
# importing module
from pymongo import MongoClient

hostname = "****"
database = "****"
port = "****"
username = "****"
password = "****"

uri = "mongodb://" + username + ":" + password + "@" + hostname + ":" + port + "/" + database

# Connect with the portnumber and host
client = MongoClient(uri)

# Access database
mydatabase = client[database]
mydatabase

Database(MongoClient(host=['1vvu1m.h.filess.io:27018'], document_class=dict, tz_aware=False, connect=True), 'OlistNoSQLDB_badtravel')

In [33]:
import pandas as pd
collection = mydatabase['product_categories']

mongo_data = pd.DataFrame(list(collection.find()))

In [34]:
mongo_data.head()
mongo_data.info()
mongo_data.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 3 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   _id                            71 non-null     object
 1   product_category_name          71 non-null     object
 2   product_category_name_english  71 non-null     object
dtypes: object(3)
memory usage: 1.8+ KB


,_id,product_category_name,product_category_name_english
count,71,71,71
unique,71,71,71
top,69a94af248fb4744617d4b4d,seguros_e_servicos,security_and_services
freq,1,1,1


In [35]:
mongo_data

,_id,product_category_name,product_category_name_english
0,69a94af248fb4744617d4b07,beleza_saude,health_beauty
1,69a94af248fb4744617d4b08,informatica_acessorios,computers_accessories
2,69a94af248fb4744617d4b09,automotivo,auto
3,69a94af248fb4744617d4b0a,cama_mesa_banho,bed_bath_table
4,69a94af248fb4744617d4b0b,moveis_decoracao,furniture_decor
...,...,...,...
66,69a94af248fb4744617d4b49,flores,flowers
67,69a94af248fb4744617d4b4a,artes_e_artesanato,arts_and_craftmanship
68,69a94af248fb4744617d4b4b,fraldas_higiene,diapers_and_hygiene
69,69a94af248fb4744617d4b4c,fashion_roupa_infanto_juvenil,fashion_childrens_clothes


In [36]:
mongo_data = mongo_data.drop(columns=['_id'])
mongo_data

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
...,...,...
66,flores,flowers
67,artes_e_artesanato,arts_and_craftmanship
68,fraldas_higiene,diapers_and_hygiene
69,fashion_roupa_infanto_juvenil,fashion_childrens_clothes


In [51]:
# Data Enriching

In [52]:
# Customers

In [53]:
# Step 1: Schema Check

In [42]:
customers_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [55]:
customers_df.show(5)

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
+--------------------+--------------------+------------------------+--------------------+--------------+
only showing top 5 rows



In [56]:
customers_df.describe().show()

26/03/05 09:37:04 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+--------------------+--------------------+------------------------+-------------------+--------------+
|summary|         customer_id|  customer_unique_id|customer_zip_code_prefix|      customer_city|customer_state|
+-------+--------------------+--------------------+------------------------+-------------------+--------------+
|  count|               99441|               99441|                   99441|              99441|         99441|
|   mean|                NULL|                NULL|       35137.47458291851|               NULL|          NULL|
| stddev|                NULL|                NULL|      29797.938996206158|               NULL|          NULL|
|    min|00012a2ce6f8dcda2...|0000366f3b9a7992b...|                    1003|abadia dos dourados|            AC|
|    max|ffffe8b65bbe3087b...|ffffd2657e2aad290...|                   99990|             zortea|            TO|
+-------+--------------------+--------------------+------------------------+-------------------+--------

In [66]:
# step 2: Drop Duplicates

customer_df = customers_df.dropDuplicates(['customer_id'])

In [67]:
customers_df.describe().show() # There were no duplicates here so rows remain same

+-------+--------------------+--------------------+------------------------+-------------------+--------------+
|summary|         customer_id|  customer_unique_id|customer_zip_code_prefix|      customer_city|customer_state|
+-------+--------------------+--------------------+------------------------+-------------------+--------------+
|  count|               99441|               99441|                   99441|              99441|         99441|
|   mean|                NULL|                NULL|       35137.47458291851|               NULL|          NULL|
| stddev|                NULL|                NULL|      29797.938996206158|               NULL|          NULL|
|    min|00012a2ce6f8dcda2...|0000366f3b9a7992b...|                    1003|abadia dos dourados|            AC|
|    max|ffffe8b65bbe3087b...|ffffd2657e2aad290...|                   99990|             zortea|            TO|
+-------+--------------------+--------------------+------------------------+-------------------+--------

In [63]:
# Step 3: Missing Values

from pyspark.sql.functions import col, sum, when

customers_df.select([sum(when(col(c).isNull(),1).otherwise(0)).alias(c) for c in customers_df.columns]).show()

+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+



In [44]:
# Orders

In [64]:
# Step 1: Schema Validation
orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [46]:
orders_df.show(5)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [68]:
# Step 2: Remove Duplicates

orders_df = orders_df.dropDuplicates(["order_id"])

In [70]:
orders_df.describe().show() # There were no duplicates here so rows remain same

+-------+--------------------+--------------------+------------+
|summary|            order_id|         customer_id|order_status|
+-------+--------------------+--------------------+------------+
|  count|               99441|               99441|       99441|
|   mean|                NULL|                NULL|        NULL|
| stddev|                NULL|                NULL|        NULL|
|    min|00010242fe8c5a6d1...|00012a2ce6f8dcda2...|    approved|
|    max|fffe41c64501cc87c...|ffffe8b65bbe3087b...| unavailable|
+-------+--------------------+--------------------+------------+



In [72]:
# Step 3: Missing Values

orders_df.select([sum(when(col(c).isNull(),1).otherwise(0)).alias(c) for c in orders_df.columns]).show()

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [76]:
orders_df.select(['order_status']).distinct().show()

+------------+
|order_status|
+------------+
|   delivered|
|    canceled|
|     created|
|     shipped|
|  processing|
|    invoiced|
| unavailable|
|    approved|
+------------+



In [83]:
orders_df.select(['order_status','order_approved_at','order_delivered_carrier_date','order_delivered_customer_date']) \
                    .filter(col('order_approved_at').isNull() & ~(col('order_status') == 'canceled')).show()

+------------+-----------------+----------------------------+-----------------------------+
|order_status|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|
+------------+-----------------+----------------------------+-----------------------------+
|   delivered|             NULL|         2017-02-22 11:23:11|          2017-03-02 11:09:19|
|   delivered|             NULL|         2017-02-22 11:23:11|          2017-03-03 18:43:43|
|   delivered|             NULL|         2017-02-22 11:42:51|          2017-03-03 12:16:03|
|     created|             NULL|                        NULL|                         NULL|
|   delivered|             NULL|         2017-02-22 16:25:25|          2017-03-01 08:07:38|
|   delivered|             NULL|         2017-02-22 11:31:06|          2017-03-02 12:06:06|
|   delivered|             NULL|         2017-02-23 09:01:52|          2017-03-02 10:05:06|
|     created|             NULL|                        NULL|                   

In [86]:
orders_df = orders_df.na.drop('all') # dropping rows where all columns are NUll

In [87]:
orders_df.describe().show()

+-------+--------------------+--------------------+------------+
|summary|            order_id|         customer_id|order_status|
+-------+--------------------+--------------------+------------+
|  count|               99441|               99441|       99441|
|   mean|                NULL|                NULL|        NULL|
| stddev|                NULL|                NULL|        NULL|
|    min|00010242fe8c5a6d1...|00012a2ce6f8dcda2...|    approved|
|    max|fffe41c64501cc87c...|ffffe8b65bbe3087b...| unavailable|
+-------+--------------------+--------------------+------------+



In [90]:
# Calculate Delivery and Time Delays
from pyspark.sql.functions import col,to_date,datediff,current_date,when

orders_df = orders_df.withColumn("actual_delivery_time", datediff("order_delivered_customer_date", "order_purchase_timestamp"))
orders_df = orders_df.withColumn("estimated_delivery_time", datediff("order_estimated_delivery_date", "order_purchase_timestamp"))
orders_df =orders_df.withColumn("Delay Time", col("actual_delivery_time") - col("estimated_delivery_time"))

DataFrame[order_id: string, customer_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, actual_delivery_time: int, estimated_delivery_time: int, Delay Time: int]

In [93]:
orders_df.select(['order_delivered_customer_date','order_purchase_timestamp','order_estimated_delivery_date','actual_delivery_time','estimated_delivery_time','Delay Time']).show()

+-----------------------------+------------------------+-----------------------------+--------------------+-----------------------+----------+
|order_delivered_customer_date|order_purchase_timestamp|order_estimated_delivery_date|actual_delivery_time|estimated_delivery_time|Delay Time|
+-----------------------------+------------------------+-----------------------------+--------------------+-----------------------+----------+
|          2018-01-22 13:19:16|     2018-01-14 14:33:31|          2018-02-05 00:00:00|                   8|                     22|       -14|
|          2017-12-18 22:03:38|     2017-12-10 11:53:48|          2018-01-04 00:00:00|                   8|                     25|       -17|
|          2018-07-09 14:04:07|     2018-07-04 12:08:27|          2018-07-25 00:00:00|                   5|                     21|       -16|
|          2018-03-29 18:17:31|     2018-03-19 18:40:33|          2018-03-29 00:00:00|                  10|                     10|         0|

In [47]:
# payments

In [49]:
payments_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)



In [50]:
payments_df.show(5)

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|
+--------------------+------------------+------------+--------------------+-------------+
only showing top 5 rows



In [94]:
# Similarly we do for all

In [95]:
# Now joining

In [97]:
final_df = orders_df.join(customers_df,"customer_id","left")

final_df = final_df.join(payments_df,"order_id","left")

final_df = final_df.join(items_df,"order_id","left")

final_df = final_df.join(products_df,"product_id","left")

final_df = final_df.join(sellers_df,"seller_id","left")

final_df = final_df.join(geolocation_df,geolocation_df.geolocation_zip_code_prefix == final_df.customer_zip_code_prefix, "left")

final_df = final_df.join(reviews_df,"order_id","left")

In [98]:
final_df.show(5)

+--------------------+--------------------+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+-----------------------+----------+--------------------+------------------------+-------------+--------------+------------------+------------+--------------------+-------------+-------------+-------------------+------+-------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+----------------------+-----------+------------+---------------------------+-------------------+-------------------+----------------+-----------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|            order_id|           seller_id|          product_id|         customer_

In [99]:
final_df.columns

['order_id',
 'seller_id',
 'product_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'actual_delivery_time',
 'estimated_delivery_time',
 'Delay Time',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'payment_sequential',
 'payment_type',
 'payment_installments',
 'payment_value',
 'order_item_id',
 'shipping_limit_date',
 'price',
 'freight_value',
 'product_category_name',
 'product_name_lenght',
 'product_description_lenght',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm',
 'seller_zip_code_prefix',
 'seller_city',
 'seller_state',
 'geolocation_zip_code_prefix',
 'geolocation_lat',
 'geolocation_lng',
 'geolocation_city',
 'geolocation_state',
 'review_id',
 'review_score',
 'review_comment_title',
 'review_comment_message',
 'review_creati

In [103]:
mongo_data

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
...,...,...
66,flores,flowers
67,artes_e_artesanato,arts_and_craftmanship
68,fraldas_higiene,diapers_and_hygiene
69,fashion_roupa_infanto_juvenil,fashion_childrens_clothes


In [106]:
mongo_spark_df = spark.createDataFrame(mongo_data)

In [107]:
mongo_spark_df.show()

+---------------------+-----------------------------+
|product_category_name|product_category_name_english|
+---------------------+-----------------------------+
|         beleza_saude|                health_beauty|
| informatica_acess...|         computers_accesso...|
|           automotivo|                         auto|
|      cama_mesa_banho|               bed_bath_table|
|     moveis_decoracao|              furniture_decor|
|        esporte_lazer|               sports_leisure|
|           perfumaria|                    perfumery|
| utilidades_domest...|                   housewares|
|            telefonia|                    telephony|
|   relogios_presentes|                watches_gifts|
|    alimentos_bebidas|                   food_drink|
|                bebes|                         baby|
|            papelaria|                   stationery|
| tablets_impressao...|         tablets_printing_...|
|           brinquedos|                         toys|
|       telefonia_fixa|     

In [108]:
final_df = final_df.join(mongo_spark_df,"product_category_name","left")

In [109]:
final_df.show(2)

+---------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+-----------------------+----------+--------------------+------------------------+-------------+--------------+------------------+------------+--------------------+-------------+-------------+-------------------+------+-------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+----------------------+-----------+------------+---------------------------+-------------------+-------------------+----------------+-----------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+-----------------------------+
|product_category_name|            order_id|        

In [110]:
final_df.write.mode("overwrite").parquet("gs://otis-dataset-project/Silver")